# Exercise 07

## Changes to the Evacuation Model

To enable exercises about sensing and interaction, the evacuation model has been modified:

* Bug fix in `agent.cooperate` (not turning if there is not more than one other agent in the room)
* Setting default values for `floor_size` (14) and `human_count` (70)
* New model and agent parameter `maxsight`
* Changes in `agent.explore_fieldofvision` to consider `maxsight`
* New switch `DISTANCE_NOISE` to add noise to the calculation of distances to exits (`* self.model.rng.normal(loc=1.0, scale=2.0)`)
* New model parameters `interact_neumann2`, `interact_moore2` and `interact_swnetwork` and agent parameter `interactionmatrix` (combining the three before-mentioned), which indicate the probability of passing alarm message via the particular topology.
* When one of `interact_neumann2`, `interact_moore2` and `interact_swnetwork` is not `None` initially only one agent knows about the alarm.
* With a probabilty of 0.1 each agent who is not alarmed gets alarmed (otherwise the room would never be evacuated if an agent is not informed).
* Adding `net.NetworkGrid`

## Evaluation Code


In [ ]:
from mesa.batchrunner import batch_run
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

import sys
sys.path.insert(0,'../../abmodel')

from fire_evacuation.model import FireEvacuation
from fire_evacuation.agent import Human


unikcolors = [np.array((80,149,200))/255, np.array((74,172,150))/255,
                                                  np.array((234,195,114))/255, np.array((199,16,92))/255]
uniks = LinearSegmentedColormap.from_list( 'unik', unikcolors)

def run_model(model_a, model_b, maxsteps = 100):
    fig = plt.figure(figsize=(12, 12))
    model_a.run(maxsteps)
    model_b.run(maxsteps)
    
    ax = fig.add_subplot(2, 1, 2)
    ax.set_xlabel("Steps")
    ax.set_ylabel("Number of escaped through the exit")
    da = model_a.datacollector.get_model_vars_dataframe()[['EscapedWest', 'EscapedSouth','EscapedEast', 'EscapedNorth']]
    db = model_b.datacollector.get_model_vars_dataframe()[['EscapedWest', 'EscapedSouth','EscapedEast', 'EscapedNorth']]
    db.columns = ['EscapedWest_B', 'EscapedSouth_B','EscapedEast_B', 'EscapedNorth_B']
    da.plot(ax=ax, colormap=uniks, ls="solid")
    db.plot(ax=ax, colormap=uniks, ls="dashed")

# Task 02 (Sensing in the evacuation model)

## Subtask 02

**Points: 10**

In which way is the agents’ sensing currently limited in the evacuation model?

**Describe your findings here**

## Subtask 03

**Points: 20**

We can change an agents sensing by limiting the field of vision, which is applied when agents search for exits and others to help. Explore the model's sensitivity towards the `maxsight` parameter with batch runs below and discuss the results (note: it may take a few minutes for batch runs to complete!). Also conduct further simulations with a higher proportion (e.g. 0.5) of agents not believing alarm (copy and adapt the code below). Are the results as you expected?


In [ ]:
from IPython.utils import io
        
params = dict(
    nervousness_mean = 0.3,
    alarm_believers_prop=0.9,
    maxsight = {2,4,6,8,10,12,14,16},
    seed = range(0,30),
)

with io.capture_output() as captured:
    results = batch_run(
            FireEvacuation,
            parameters=params,
            data_collection_period = 1,
            iterations = 1,
            max_steps = 500,
        )
data = pd.DataFrame(results)[['maxsight', "TurnCount", 'Step', 'seed']].round(decimals=1)
db = data.groupby(['seed','maxsight']).agg(np.max).groupby(['maxsight']).agg(np.mean)
db

**Describe your finding here (300 words max.)!**

## Subtask 04

**Points: 15**

Sensing is often not 100% accurate - uncertainty is involved. A way to represent uncertainty in agent-based modelling it to introduce noise - a random factor that reduces accuracy of the perceived value. In the model, uncertainty has been introduced for sensing the distance between the agent and a certain exit:

In [ ]:
from IPython.utils import io
        
params = dict(
    nervousness_mean = 0.3,
    distancenoise = {False,True},
    seed = range(0,20),
)

with io.capture_output() as captured:
    results = batch_run(
            FireEvacuation,
            parameters=params,
            data_collection_period = 1,
            iterations = 1,
            max_steps = 500,
        )
data = pd.DataFrame(results)[['distancenoise', 'Step', 'seed']].round(decimals=1)
db = data.groupby(['seed','distancenoise']).agg(np.max).groupby(['distancenoise']).agg(np.mean)
db

Find two more processes of agents' sensing that could be modelled with uncertainty. How would you introduce uncertainty (provide pseudo code)? Discuss whether you should add this option to the model code!

**Describe your ideas here (250 words max. and pseudo code)**

# Task 02 (Exploring interaction in the evacuation model)

## Subtask 01

**Points: 20**

A new option was added to the model: Assume the fire alarm is broken or not existing, and a single, randomly choosen agent gets aware of an incident in the room that requires all to escape. We investigate different ways of interaction in terms of passing the information of fire alarm:

* Propagation in the von-Neumann-neighbourhood
* Propagation in the Moore-Neighbourhood
* Propagation on a Small-World-Network (Watts-Beta)

The model parameters determine the probability, by which the information is passed from an informed agent to its neighbours on the particular topology.

Execute the following three code blocks and compare the results! For small-world networks, why is the difference between low values of propagation probability higher than for higher values of the probability?

In [ ]:
from IPython.utils import io
        
params = dict(
    nervousness_mean = 0.3,
    cooperation_mean = 0.1,
    interact_neumann = {0.01,0.02,0.05,0.1,0.5,1.0},
    interact_moore = 0.0,
    interact_swnetwork = 0.0,
    seed = range(0,30),
)

with io.capture_output() as captured:
    results = batch_run(
            FireEvacuation,
            parameters=params,
            data_collection_period = 1,
            iterations = 1,
            max_steps = 500,
        )
data = pd.DataFrame(results)[['interact_neumann', 'interact_moore','interact_swnetwork', 'Step', 'seed']].round(decimals=2)
db = data.groupby(['seed','interact_neumann','interact_moore','interact_swnetwork',]).agg(np.max).groupby(['interact_neumann','interact_moore','interact_swnetwork']).agg(np.mean)
db

In [ ]:
from IPython.utils import io
        
params = dict(
    nervousness_mean = 0.3,
    cooperation_mean = 0.1,
    interact_neumann = 0.0,
    interact_moore = {0.01,0.02,0.05,0.1,0.5,1.0},
    interact_swnetwork = 0.0,
    seed = range(0,30),
)

with io.capture_output() as captured:
    results = batch_run(
            FireEvacuation,
            parameters=params,
            data_collection_period = 1,
            iterations = 1,
            max_steps = 500,
        )
data = pd.DataFrame(results)[['interact_neumann', 'interact_moore','interact_swnetwork', 'Step', 'seed']].round(decimals=2)
db = data.groupby(['seed','interact_neumann','interact_moore','interact_swnetwork',]).agg(np.max).groupby(['interact_neumann','interact_moore','interact_swnetwork']).agg(np.mean)
db

In [ ]:
from IPython.utils import io
        
params = dict(
    nervousness_mean = 0.3,
    cooperation_mean = 0.3,
    interact_neumann = 0.0,
    interact_moore = 0.0,
    interact_swnetwork = {0.01,0.02,0.05,0.1,0.5,1.0},
    seed = range(0,30),
)

with io.capture_output() as captured:
    results = batch_run(
            FireEvacuation,
            parameters=params,
            data_collection_period = 1,
            iterations = 1,
            max_steps = 500,
        )
data = pd.DataFrame(results)[['interact_neumann', 'interact_moore','interact_swnetwork', 'Step', 'seed']].round(decimals=2)
db = data.groupby(['seed','interact_neumann','interact_moore','interact_swnetwork',]).agg(np.max).groupby(['interact_neumann','interact_moore','interact_swnetwork']).agg(np.mean)
db

**Place your answer here (max 500 words)**

## Subtask 02

**Points: 15**

The position of the initial knowing agent matters when the propagation probabilties are in a certain range. Does this range covers rather low probability values or rather high probability values? Explain! Does this represent what would happen in reality?

Which position would be ideal for each single interaction topology (vonNeumann, Moore, Network)? Format your answer either as list or table.

**Write your answer here (<200 words)!**

## Subtask 03

**Points: 15**

Implement placing the initial knowing agent at a beneficial position for propagation on the smallworld-network. Consider [this answer](https://stackoverflow.com/a/58392749/3957413)!

First describe your idea here in pseudo code, then implement in `model.py` at line 274ff.

**State your pseudo code here**